# 03 · Model Comparison: Which model should we use?

Before building our main model, we test **9 different models** on the same data and the same split, and compare them fairly.
We use the **original inputs** here (no extra features yet), so every model starts from the same place.

What a good model needs for *our* problem:
1. **Rank customers well** (who is most likely to leave): measured by ROC-AUC
2. **Find leavers in a small contact list**: % of real leavers found in the top 20% riskiest
3. **Work on new customers**, not just memorise old ones: small train vs test gap
4. **Be stable**: similar score on every cross-validation round
5. **Be explainable**: can we tell a manager *why* a customer is at risk?
6. **Support what-if simulation**: does changing one detail (e.g. contract) change the risk in a sensible way?
7. **Be fast and simple to run in the app**

> Run `01_data_types.ipynb` first. Next: notebook 04 builds the chosen model step by step,
> and notebook 05 improves and tunes it.

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # hide a harmless SVM notice from newer scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                              AdaBoostClassifier)
from sklearn.neural_network import MLPClassifier

df = pd.read_pickle("telco_clean.pkl")
category_columns = df.select_dtypes(include="category").columns

def encode(customers, columns=None):
    # original inputs: drop unused columns and turn text into 0/1 columns
    X = customers.drop(columns=["Churn", "customerID", "TotalCharges"])
    for col in category_columns:
        # remind each column of all its possible values, so encoding is always the same
        X[col] = pd.Categorical(X[col], categories=df[col].cat.categories)
    X = pd.get_dummies(X, drop_first=True, dtype=int)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0)
    return X

train, test = train_test_split(df, test_size=0.2, stratify=df["Churn"], random_state=42)

X_train = encode(train)
X_test = encode(test, columns=list(X_train.columns))
y_train, y_test = train["Churn"], test["Churn"]
print("Train:", X_train.shape, "| Test:", X_test.shape)

## The contenders

| Model | Idea in one line |
|---|---|
| **Logistic Regression** | gives each factor a weight, adds them up, turns the total into a probability |
| Decision Tree | a flowchart of yes/no questions |
| K-Nearest Neighbours (KNN) | looks at the 50 most similar past customers: did they leave? |
| Naive Bayes | uses probability rules, assuming all factors are independent |
| Support Vector Machine (SVM) | draws the best curved boundary between leavers and stayers |
| Random Forest | 300 decision trees vote |
| Gradient Boosting | trees built one after another, each fixing the last one's mistakes |
| AdaBoost | many tiny trees, focusing more on customers it got wrong |
| Neural Network (MLP) | layers of connected "neurons" that learn patterns |

Models that care about the size of numbers (LogReg, KNN, SVM, Neural Net) get a `StandardScaler` in front.
Settings are sensible defaults, kept simple on purpose.

In [ ]:
def scaled(model):
    return make_pipeline(StandardScaler(), model)

models = {
    "Logistic Regression": scaled(LogisticRegression(class_weight="balanced", max_iter=2000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=20,
                                            class_weight="balanced", random_state=42),
    "KNN": scaled(KNeighborsClassifier(n_neighbors=50)),
    "Naive Bayes": GaussianNB(),
    "SVM": scaled(SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=10,
                                            class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=3,
                                                        class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=42),
    "Neural Network": scaled(MLPClassifier(hidden_layer_sizes=(32, 16), alpha=0.01,
                                           max_iter=500, early_stopping=True, random_state=42)),
}

## How we score them

- **cv AUC:** average ROC-AUC over 5 cross-validation rounds on the training customers (fair, uses no test data)
- **test AUC:** ROC-AUC on the 1,409 unseen test customers
- **top-20% capture:** if we contact only the 20% riskiest test customers, what % of the real leavers do we reach?
  This is exactly how the retention team will use the model.
- **overfit gap:** train AUC minus test AUC (big gap = memorising)
- **train time:** seconds to train once

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def top20_capture(true, prob):
    # % of real leavers found in the 20% of customers with the highest risk
    order = np.argsort(-prob)
    top = order[: int(len(prob) * 0.20)]
    return true.to_numpy()[top].sum() / true.sum() * 100

rows = []
fold_scores = {}
for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=folds, scoring="roc_auc")
    fold_scores[name] = cv

    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    train_prob = model.predict_proba(X_train)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]

    rows.append({
        "model": name,
        "cv AUC": round(cv.mean(), 4),
        "cv spread (+/-)": round(cv.std(), 4),
        "test AUC": round(roc_auc_score(y_test, test_prob), 4),
        "top-20% capture %": round(top20_capture(y_test, test_prob), 1),
        "overfit gap": round(roc_auc_score(y_train, train_prob) - roc_auc_score(y_test, test_prob), 4),
        "train time (s)": round(train_time, 2),
    })
    print(f"done: {name}")

results = pd.DataFrame(rows).sort_values("cv AUC", ascending=False).reset_index(drop=True)
results

## Chart 1 · Ranking quality (cv AUC with spread)

The black lines show how much each model's score moves between cross-validation rounds.
If two bars overlap within those lines, the difference between them is **just noise**.

In [ ]:
plot = results.sort_values("cv AUC")
colors = ["darkorange" if m == "Logistic Regression" else "steelblue" for m in plot["model"]]

plt.figure(figsize=(8, 5))
plt.barh(plot["model"], plot["cv AUC"], xerr=plot["cv spread (+/-)"], color=colors, capsize=4)
plt.xlim(0.75, 0.88)
plt.xlabel("cv ROC-AUC (higher = better ranking)")
plt.title("Model comparison (orange = Logistic Regression)")
plt.show()

## Chart 2 · Score in every cross-validation round

Each line is one model. Models that stay on top in **every** round are reliably better; lines that cross are basically tied.

In [ ]:
plt.figure(figsize=(9, 5))
for name, scores in fold_scores.items():
    style = dict(linewidth=3, color="darkorange") if name == "Logistic Regression" else dict(alpha=0.6)
    plt.plot(range(1, 6), scores, marker="o", label=name, **style)
plt.xticks(range(1, 6))
plt.xlabel("Cross-validation round")
plt.ylabel("ROC-AUC")
plt.title("AUC in each round")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

## Is the best model *really* better than Logistic Regression?

We count in how many of the 5 rounds each model beat Logistic Regression, and by how much on average.

In [ ]:
lr_scores = fold_scores["Logistic Regression"]

rows = []
for name, scores in fold_scores.items():
    if name == "Logistic Regression":
        continue
    diff = scores - lr_scores
    rows.append([name, int((diff > 0).sum()), round(diff.mean(), 4)])

pd.DataFrame(rows, columns=["model", "rounds it beat LogReg (of 5)", "average AUC difference"]) \
    .sort_values("average AUC difference", ascending=False)

## Test 6 · Does the model react sensibly in a what-if?

Our retention plan depends on asking *"what if this customer moved to a yearly contract?"*
We take every month-to-month test customer, switch them to "One year", and check how many get **lower** risk (as they should).

In [ ]:
m2m = test[test["Contract"] == "Month-to-month"]
changed = m2m.copy()
changed["Contract"] = "One year"

X_before = encode(m2m, columns=list(X_train.columns))
X_after = encode(changed, columns=list(X_train.columns))

rows = []
for name, model in models.items():
    before = model.predict_proba(X_before)[:, 1]
    after = model.predict_proba(X_after)[:, 1]
    rows.append([name,
                 round((after < before).mean() * 100, 1),
                 round((after == before).mean() * 100, 1),
                 round((after > before).mean() * 100, 1)])

whatif = pd.DataFrame(rows, columns=["model", "risk went DOWN %", "no change %", "risk went UP %"])
whatif

**How to read it:** a yearly contract should lower the risk for (almost) everyone.
- If a model says "no change" for many customers, it can't be used to plan retention offers
- If it says the risk goes **up** for some, the what-if results would be confusing to explain

## Explainability and practical scorecard

In [ ]:
practical = pd.DataFrame({
    "model": ["Logistic Regression", "Decision Tree", "KNN", "Naive Bayes", "SVM",
              "Random Forest", "Gradient Boosting", "AdaBoost", "Neural Network"],
    "explain WHY to a manager": ["Easy: weight / odds ratio per factor", "Easy: flowchart, but only a few rules",
                                 "Hard: 'similar customers left'", "Medium: probabilities per factor",
                                 "Hard: no weights to show", "Hard: 300 trees, needs extra tools",
                                 "Hard: needs extra tools (e.g. SHAP)", "Hard: 200 small trees",
                                 "Hard: a black box"],
    "app / deployment": ["Tiny model, instant", "Tiny, instant", "Must keep all training customers",
                         "Tiny, instant", "Slow to predict probabilities", "Large file",
                         "Medium", "Medium", "Medium, sensitive to settings"],
})
practical

## Final scorecard

In [ ]:
final = results.merge(whatif[["model", "risk went DOWN %"]], on="model").merge(practical, on="model")
final[["model", "cv AUC", "test AUC", "top-20% capture %", "overfit gap",
       "risk went DOWN %", "explain WHY to a manager"]]

## Decision: Logistic Regression

### Results (original inputs)

| Model | cv AUC | test AUC | Top-20% capture | Overfit gap | What-if: risk down |
|---|---|---|---|---|---|
| Gradient Boosting | 0.848 | 0.845 | 50.0% | 0.028 | 100% |
| AdaBoost | 0.848 | 0.843 | 50.3% | 0.011 | 100% |
| Random Forest | 0.847 | 0.843 | 50.5% | 0.052 | 99.7% |
| **Logistic Regression** | **0.845** | **0.838** | **49.5%** | **0.010** | **100%** |
| Neural Network | 0.837 | 0.842 | 48.7% | 0.018 | 99.1% |
| KNN | 0.830 | 0.821 | 45.7% | 0.021 | 92.4% |
| SVM | 0.829 | 0.819 | 48.1% | 0.057 | 99.0% |
| Decision Tree | 0.826 | 0.831 | 45.2% | 0.014 | 97.4% |
| Naive Bayes | 0.820 | 0.807 | 46.8% | 0.014 | 100% |

### What this tells us (honestly)
- The **tree ensembles are slightly ahead**: Gradient Boosting and AdaBoost beat Logistic Regression in 4 of 5 rounds,
  by about **0.0035 AUC** on average. That's small, and in practice it means about 1 extra leaver in the top-20% list
  (50.0% vs 49.5% capture).
- KNN, SVM, Naive Bayes and a single Decision Tree are clearly worse.

### Why we still choose Logistic Regression
1. **Accuracy is practically tied** with the best models (0.845 vs 0.848). The gap is a fraction of a point.
2. **Least overfitting:** train vs test gap 0.010, versus 0.028 for Gradient Boosting and 0.052 for Random Forest.
3. **Reliable what-if simulation:** a yearly contract lowered the risk for **100%** of customers, so our retention
   simulation gives clear, explainable answers.
4. **Explainable:** every factor gets a weight and an odds ratio ("a two-year contract cuts the odds of leaving by about 76%").
   The tree ensembles need extra tools (like SHAP) and are still harder to present.
5. **Simple and fast:** trains in 0.01 seconds, tiny model, instant in the app.
6. **Room to improve:** a simple model gains the most from good features. In notebook 05, engineered features and
   tuning lift Logistic Regression to **0.850 cv AUC**, above every untuned model here.

### One line for judges
> "We compared 9 models. The tree ensembles were a fraction of a point better at first, but Logistic Regression overfit
> less, explained its decisions, behaved sensibly in what-if tests, and after feature engineering and tuning it caught up and passed them."
